# Ch.2 演習ノートブック：データ可視化・EDA

| 項目 | 内容 |
|------|------|
| **使用データ** | ワインデータ（`load_wine()`）178件・13特徴量・3品種 |
| **作るもの** | 仮説付き EDA グラフセット（3枚）＋ 仮説シート 1枚 |
| **実務での対応物** | 分析報告資料・ステークホルダー向けインサイトスライド |
| **時間の目安** | 演習 40分（問3まで完了で十分） |

## このノートブックの使い方

**EDA のゴールは「グラフを作ること」ではなく「仮説を言葉にすること」です。**

- グラフのコードは **Copilot に任せてOK**
- 「何を見たいか」と「グラフから何が読めるか」は **自分で考える**

> ✅ **問3まで完了すれば十分です。**

---
## 🤖 Copilot Chat の使い方

### 開き方：ウェブブラウザで開く

```
Microsoft Copilot → https://copilot.microsoft.com
GitHub Copilot   → https://github.com/copilot
```

> 画面左半分に JupyterLab、右半分にブラウザ（Copilot Chat）を並べて使ってください。  
> JupyterLab のサイドバー機能は使いません。

### 演習のコードはすべて Copilot Chat に書いてもらう

このコースでは **演習コードはすべて Copilot Chat に生成してもらうことを推奨** します。  
自分の仕事は「**何をしたいかを言語化すること**」と「**結果を解釈すること**」です。

### 質問の型（必ずこの形式でコピペして使う）

```
【やりたいこと】〇〇したい
【使うライブラリ】pandas, matplotlib など
【データの形】DataFrame で列名は〇〇、行数は〇〇件
【環境】Python 3.8.6、Windows、JupyterLab
【わからない点】〇〇の書き方 / 〇〇でエラーが出た
```

### Copilot 活用ルール

| タイミング | ルール |
|-----------|--------|
| 座学中 | **禁止** ― まず概念を頭に入れる |
| 演習 問1〜2 | **上の型でプロンプトを書いて質問する** |
| 演習 問3（考察） | コードはOK・**考察文は自分で書く** |
| 問4（発展） | **自由に活用** |

> 詳細テンプレート集 → `07_copilot_prompt_guide.md`

---
## 🔧 準備

### ライブラリを読み込む

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
try:
    import japanize_matplotlib   # グラフの日本語表示を有効にする
except ImportError:
    pass

from sklearn.datasets import load_wine

print("✅ ライブラリ読み込み完了")

### データを読み込む

In [ ]:
wine = load_wine()
df   = pd.DataFrame(wine.data, columns=wine.feature_names)
df["品種"] = wine.target
species = {0: "Barolo", 1: "Grignolino", 2: "Barbera"}
df["品種名"] = df["品種"].map(species)

# カラーパレット（品種ごとの色）
palette = {"Barolo": "#2D4A7A", "Grignolino": "#E86A2C", "Barbera": "#27AE60"}

print(f"✅ データ準備完了  {len(df)}件・{df.shape[1]}列")

---
## 📋 グラフ選びの考え方（座学デモ再現）

### 「何を見たいか」でグラフが決まる

| 見たいこと | グラフ | コード |
|-----------|--------|--------|
| 1変数の分布 | ヒストグラム | `plt.hist()` |
| グループ間の比較 | 箱ひげ図 | `df.boxplot()` |
| 2変数の関係 | 散布図 | `plt.scatter()` |
| 全変数の相関を一覧 | ヒートマップ | `sns.heatmap()` |

**実務での使い分け：**
- 売上分布を確認したい → ヒストグラム
- 地域ごとの売上差を確認したい → 箱ひげ図
- 気温と売上の関係を見たい → 散布図
- どの変数が購買に関係しているか → ヒートマップ

### ヒストグラムで1変数の分布を確認する

In [ ]:
# アルコール度数の分布
plt.figure(figsize=(8, 4))
plt.hist(df["alcohol"], bins=20, color="#2D4A7A", edgecolor="white")
plt.xlabel("アルコール度数")
plt.ylabel("件数")
plt.title("アルコール度数の分布（全品種）")
plt.tight_layout()
plt.show()

### 品種ごとに重ねて表示する（半透明で重ねる）

In [ ]:
plt.figure(figsize=(8, 4))
for 品種名, color in palette.items():
    subset = df[df["品種名"] == 品種名]
    plt.hist(subset["alcohol"], bins=15, alpha=0.6, label=品種名, color=color)
plt.xlabel("アルコール度数")
plt.ylabel("件数")
plt.title("品種別アルコール度数の分布")
plt.legend()
plt.tight_layout()
plt.show()

### 箱ひげ図でグループ差を確認する

**実務での対応：** 「地域ごとの売上のばらつき」を1枚で比較するのと同じです。

In [ ]:
plt.figure(figsize=(7, 5))
df.boxplot(column="proline", by="品種名", figsize=(7, 5))
plt.title("品種別 プロリン含有量の分布")
plt.suptitle("")   # デフォルトのタイトルを非表示
plt.xlabel("品種")
plt.ylabel("プロリン含有量")
plt.tight_layout()
plt.show()

### 相関ヒートマップで全変数を一覧確認する

In [ ]:
# 相関行列を計算（品種列は数値だが意味が異なるので除外）
corr = df.drop(columns=["品種", "品種名"]).corr()

plt.figure(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt=".1f", cmap="coolwarm", center=0,
            square=True, linewidths=0.4, annot_kws={"size": 9})
plt.title("特徴量間の相関行列")
plt.tight_layout()
plt.show()

#### 観察ポイント
- **赤い（r > 0.6）** マス ← 正の相関が強い変数ペア。一方が増えると他方も増える
- **青い（r < -0.4）** マス ← 負の相関。一方が増えると他方は減る
- この情報をもとに問2の散布図の変数を選びましょう

---
## 問1：品種ごとの平均アルコール度数を棒グラフで比較する ★☆☆

**実務での対応：** 四半期ごとの売上平均を棒グラフで経営会議に報告するのと同じ形式です。

### 平均値を集計する（Ch.1 の復習）
まず `groupby()` で品種ごとの平均を出します。

In [ ]:
# 品種ごとのアルコール度数の平均
mean_alc = df.groupby("品種名")["alcohol"].mean().round(2)
print(mean_alc)

### 棒グラフで可視化する

In [ ]:
plt.figure(figsize=(7, 5))
bars = plt.bar(mean_alc.index, mean_alc.values,
               color=[palette[k] for k in mean_alc.index])
plt.xlabel("品種")
plt.ylabel("平均アルコール度数")
plt.title("品種別 平均アルコール度数")
plt.ylim(11, 15)   # Y軸の範囲を調整して差を見やすくする

# バーの上に数値を表示
for bar, val in zip(bars, mean_alc.values):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
             f"{val:.2f}", ha="center", va="bottom", fontsize=11)

plt.tight_layout()
plt.show()

#### ✅ チェックポイント
- 3品種の棒グラフが表示されましたか？
- どの品種が最も高いですか？差は何度くらいありますか？
- この差は「大きい」ですか？「小さい」ですか？（全体の値域 11〜15 と比較して）

> 🤖 **Copilot Chat プロンプト**（右サイドバーのチャット欄にコピーして使ってください）
>  （詳細テンプレート：`07_copilot_prompt_guide.md` の [P2-1]）

> ```
> 【やりたいこと】proline（プロリン含有量）の品種別平均を棒グラフで表示したい
> 【使うライブラリ】matplotlib, pandas, japanize_matplotlib
> 【データの形】DataFrame で「品種名」列と「proline」列がある、178件
> 【環境】Python 3.8.6、Windows、JupyterLab
> 【わからない点】groupby().mean().plot(kind='bar') の書き方と、バーに数値を表示する方法
> ```

---
## 問2：2変数を選んで品種別散布図を作る ★★☆

**実務での対応：** 「広告費と売上の関係」「気温とアイスの売上」など、2変数の関係を可視化する作業です。

### 変数を選ぶ（ヒートマップを参考に）
ヒートマップで **赤いマス（相関が高い）** のペアを選んでください。

推奨ペア例：
- `alcohol` と `proline`
- `color_intensity` と `od280/od315_of_diluted_wines`

選んだ変数を下のセルの `x_col` / `y_col` に入れてください。

In [ ]:
# ← ここを変えてみてください
x_col = "alcohol"
y_col = "proline"

### 基本散布図（色分けなし）で確認する

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(df[x_col], df[y_col], alpha=0.6, color="#7A9CC0")
plt.xlabel(x_col)
plt.ylabel(y_col)
plt.title(f"{x_col} vs {y_col}")
plt.tight_layout()
plt.show()

### 品種ごとに色分けして表示する

In [ ]:
plt.figure(figsize=(7, 5))
for 品種名, color in palette.items():
    subset = df[df["品種名"] == 品種名]
    plt.scatter(subset[x_col], subset[y_col],
                label=品種名, color=color, alpha=0.75, edgecolors="white", linewidths=0.5)
plt.xlabel(x_col)
plt.ylabel(y_col)
plt.title(f"{x_col} vs {y_col}（品種別）")
plt.legend(title="品種")
plt.tight_layout()
plt.show()

#### ✅ チェックポイント
- 3品種が色分けされて表示されましたか？
- 品種ごとにまとまって分布していますか？それとも混在していますか？
- 「この2変数で品種を区別できそうか？」をグラフから判断してみてください

> 🤖 **Copilot Chat プロンプト**（右サイドバーのチャット欄にコピーして使ってください）
>  （詳細テンプレート：`07_copilot_prompt_guide.md` の [P2-3]）

> ```
> 【やりたいこと】alcohol と proline の散布図に回帰直線を追加したい
> 【使うライブラリ】seaborn, matplotlib, japanize_matplotlib
> 【データの形】DataFrame で「alcohol」「proline」「品種名」列がある、178件
> 【環境】Python 3.8.6、Windows、JupyterLab
> 【わからない点】sns.regplot() または sns.lmplot() の使い方と品種別色分けの方法
> ```

---
## 問3：グラフから仮説を立てる ★★☆（最重要）

**実務での対応：** 分析報告書の「インサイト（洞察）」セクションを書く作業です。

### なぜ仮説の言語化が最重要か
グラフを作るだけなら自動化できます。  
「グラフから何が言えるか」を言葉にできる人が、データサイエンティストとして価値を発揮できます。

この仮説は **Ch.4 でモデルを作ったときの答え合わせ** になります。

### 良い仮説の書き方
```
「〇〇という変数は、△△という観察から、品種の分類に効くと思う。
 理由：□□品種だけ値の範囲が他品種と重ならないから」
```

### 仮説記入欄

**問2で選んだ変数：**

> x軸：　　　　　　　　　y軸：　　　　　　

**散布図から気づいたこと：**

> （例：Barolo だけ右上の領域に集まっており、他の2品種と明確に分離されている）

**立てた仮説：**

> （例：`proline` と `alcohol` の2変数の組み合わせで Barolo を分類できると思う。  
> 散布図で Barolo だけが右上に孤立して分布しているから）

---
> 💡 **ここは Copilot に書いてもらわないでください。**  
> 自分の言葉で書いた仮説と Ch.4 の「特徴量重要度」を照らし合わせることが、今日最大の学びです。

---
## 問4（発展）：pairplot で全変数の関係を一覧する ★★★

### pairplot とは
全特徴量ペアの散布図を一気に表示します。  
「どの変数ペアが最も品種をきれいに分離できているか」を視覚的に探せます。

> ⚠️ 全13変数だと時間がかかるので、気になる変数を4〜5個に絞って実行してください。

In [ ]:
# 気になる変数を選んで pairplot を実行する（4〜5個推奨）
selected_cols = ["alcohol", "proline", "color_intensity", "flavanoids", "品種名"]

g = sns.pairplot(df[selected_cols], hue="品種名", palette=palette,
                 plot_kws={"alpha": 0.6}, diag_kind="hist")
g.fig.suptitle("品種別 pairplot（変数間の関係一覧）", y=1.02)
plt.tight_layout()
plt.show()

> 🤖 **Copilot Chat プロンプト**（右サイドバーのチャット欄にコピーして使ってください）
>  （詳細テンプレート：`07_copilot_prompt_guide.md` の [P2-1]）

> ```
> 【やりたいこと】品種ごとのalcohol分布を violinplot で表示したい
> 【使うライブラリ】seaborn, matplotlib, japanize_matplotlib
> 【データの形】DataFrame で「品種名」「alcohol」列がある、178件
> 【環境】Python 3.8.6、Windows、JupyterLab
> 【わからない点】sns.violinplot() の使い方と箱ひげ図との違い
> ```